<a href="https://colab.research.google.com/github/Yap29Kang/TMS4013-Parallel-Merge-Sort/blob/main/CUDA/Cuda_Program.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Verify GPU is active
!nvidia-smi

Tue Nov 24 08:35:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   60C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Verify CUDA is active
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Aug_20_01:58:59_PM_PDT_2025
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0


In [25]:
%%writefile CUDA_mergesort.cu

#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

// CUDA configuration
#define RUN 2
#define ITEMS 2
#define THREADS_PER_BLOCK 256

// Check CUDA errors
#define CUDA_CHECK(call) do { \
    cudaError_t err = call; \
    if (err != cudaSuccess) { \
        fprintf(stderr, "CUDA error: %s\n", cudaGetErrorString(err)); \
        exit(EXIT_FAILURE); \
    } \
} while (0)

Overwriting CUDA_mergesort.cu


In [26]:
%%writefile -a CUDA_mergesort.cu

// -------------------------------------
// STEP 1: Sort small portions
// -------------------------------------

__global__ void insertionSortKernel(int *data, int n) {

    int tid = blockIdx.x * blockDim.x + threadIdx.x;

    int start = tid * RUN;

    if (start >= n) {
        return;
    }

    int len = (start + RUN <= n) ? RUN : n - start;

    int local[RUN];

    // Copy elements into local array
    for (int i = 0; i < len; i++) {
        local[i] = data[start + i];
    }

    // Insertion sort
    for (int i = 1; i < len; i++) {

        int key = local[i];
        int j = i - 1;

        while (j >= 0 && local[j] > key) {
            local[j + 1] = local[j];
            j--;
        }

        local[j + 1] = key;
    }

    // Copy sorted elements back
    for (int i = 0; i < len; i++) {
        data[start + i] = local[i];
    }
}

Appending to CUDA_mergesort.cu


In [27]:
%%writefile -a CUDA_mergesort.cu

// -------------------------------------
// STEP 2: Co-ranking function
// -------------------------------------

__device__ int coRank(
    int k, const int *A, int m,
    const int *B, int n
) {

    int low = (k - n > 0) ? k - n : 0;
    int high = (k < m) ? k : m;

    while (low <= high) {

        int i = (low + high) / 2;
        int j = k - i;

        if (i > 0 && j < n && A[i - 1] > B[j]) {
            high = i - 1;
        }
        else if (j > 0 && i < m && B[j - 1] >= A[i]) {
            low = i + 1;
        }
        else {
            return i;
        }
    }

    return low;
}

Appending to CUDA_mergesort.cu


In [28]:
%%writefile -a CUDA_mergesort.cu

// -------------------------------------
// STEP 3: Merge Path kernel
// -------------------------------------

__global__ void mergePathKernel(
    int *data, int *temp, int n, int width
) {

    int tid = blockIdx.x * blockDim.x + threadIdx.x;

    int chunksPerPair = (2 * width + ITEMS - 1) / ITEMS;

    int pair = tid / chunksPerPair;
    int chunk = tid % chunksPerPair;

    int start = pair * (2 * width);

    if (start >= n) {
        return;
    }

    int mid = (start + width < n) ? start + width : n;
    int end = (start + 2 * width < n) ? start + 2 * width : n;

    const int *A = data + start;
    const int *B = data + mid;

    int m = mid - start;
    int lenB = end - mid;

    int outStart = chunk * ITEMS;
    int total = m + lenB;

    if (outStart >= total) {
        return;
    }

    int outEnd = (outStart + ITEMS < total)
                 ? outStart + ITEMS : total;

    int i = coRank(outStart, A, m, B, lenB);
    int j = outStart - i;

    int iEnd = coRank(outEnd, A, m, B, lenB);
    int jEnd = outEnd - iEnd;

    int k = start + outStart;

    // Merge assigned output elements
    while (i < iEnd && j < jEnd) {

        if (A[i] <= B[j]) {
            temp[k++] = A[i++];
        }
        else {
            temp[k++] = B[j++];
        }
    }

    while (i < iEnd) {
        temp[k++] = A[i++];
    }

    while (j < jEnd) {
        temp[k++] = B[j++];
    }
}

Appending to CUDA_mergesort.cu


In [29]:
%%writefile -a CUDA_mergesort.cu

// -------------------------------------
// STEP 4: Check whether array is sorted
// -------------------------------------

int isSorted(int *data, int n) {

    for (int i = 1; i < n; i++) {
        if (data[i - 1] > data[i]) {
            return 0;
        }
    }

    return 1;
}

Appending to CUDA_mergesort.cu


In [30]:
%%writefile -a CUDA_mergesort.cu

// -------------------------------------
// STEP 5: Main function
// -------------------------------------

int main() {

    int N;

    printf("Enter number of elements: ");

    if (scanf("%d", &N) != 1 || N <= 0) {
        printf("Invalid array size!\n");
        return 1;
    }

    int *h_data = (int *)malloc(N * sizeof(int));

    if (h_data == NULL) {
        printf("CPU memory allocation failed!\n");
        return 1;
    }

    // Create CUDA events
    cudaEvent_t start, stop;

    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));

    if (h_data == NULL) {
        printf("CPU memory allocation failed!\n");
        return 1;
    }

    // Generate random input values
    srand(42);

    for (int i = 0; i < N; i++) {
        h_data[i] = rand() % 100000;
    }

    // Allocate GPU memory
    int *d_data;
    int *d_temp;

    CUDA_CHECK(cudaMalloc(&d_data, N * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_temp, N * sizeof(int)));

    // Copy input array to GPU
    CUDA_CHECK(cudaMemcpy(
        d_data, h_data, N * sizeof(int),
        cudaMemcpyHostToDevice
    ));

    // -------------------------------------
    // STEP 6: Sort small portions
    // -------------------------------------

    int totalThreads = (N + RUN - 1) / RUN;

    int blocks = (totalThreads + THREADS_PER_BLOCK - 1)
                 / THREADS_PER_BLOCK;

    // Record GPU sorting start time
    CUDA_CHECK(cudaEventRecord(start));

    insertionSortKernel<<<blocks, THREADS_PER_BLOCK>>>(d_data, N);

    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());


    // -------------------------------------
    // STEP 7: Repeated Merge Path merging
    // -------------------------------------

    for (int width = RUN; width < N; width *= 2) {

        int chunksPerPair = (2 * width + ITEMS - 1) / ITEMS;

        int pairs = (N + 2 * width - 1) / (2 * width);

        totalThreads = pairs * chunksPerPair;

        blocks = (totalThreads + THREADS_PER_BLOCK - 1)
                / THREADS_PER_BLOCK;

        mergePathKernel<<<blocks, THREADS_PER_BLOCK>>>(
            d_data, d_temp, N, width
        );

        CUDA_CHECK(cudaGetLastError());
        CUDA_CHECK(cudaDeviceSynchronize());

        // Swap input and output arrays
        int *swap = d_data;
        d_data = d_temp;
        d_temp = swap;
    }

    // -------------------------------------
    // Measure total GPU sorting time
    // -------------------------------------

    CUDA_CHECK(cudaEventRecord(stop));

    CUDA_CHECK(cudaEventSynchronize(stop));

    float gpuTime = 0.0f;

    CUDA_CHECK(cudaEventElapsedTime(&gpuTime, start, stop));

    printf("\nGPU sorting time: %.4f ms\n", gpuTime);


    // -------------------------------------
    // STEP 8: Copy result back to CPU
    // -------------------------------------

    CUDA_CHECK(cudaMemcpy(
        h_data, d_data, N * sizeof(int),
        cudaMemcpyDeviceToHost
    ));

    // -------------------------------------
    // STEP 9: Verify sorting result
    // -------------------------------------

    printf("\nSorting %d elements using CUDA Merge Sort...\n", N);

    if (isSorted(h_data, N)) {
        printf("Array sorted successfully!\n");
    }
    else {
        printf("Sorting failed!\n");
    }

    // Display first 20 sorted elements
    printf("\nFirst %d sorted elements:\n", (N < 20) ? N : 20);

    for (int i = 0; i < N && i < 20; i++) {
        printf("%d ", h_data[i]);
    }

    printf("\n");

    // -------------------------------------
    // STEP 10: Free memory
    // -------------------------------------

    CUDA_CHECK(cudaFree(d_data));
    CUDA_CHECK(cudaFree(d_temp));

    free(h_data);


    // Destroy CUDA events
    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));
    return 0;
}


Appending to CUDA_mergesort.cu


In [31]:
!nvcc CUDA_mergesort.cu -o CUDA_mergesort && ./CUDA_mergesort

Enter number of elements: 10000

GPU sorting time: 0.3933 ms

Sorting 10000 elements using CUDA Merge Sort...
Array sorted successfully!

First 20 sorted elements:
23 32 39 48 61 74 77 90 98 103 124 141 145 158 161 169 214 218 220 229 
